In [ ]:
import jax
jax.config.update('jax_enable_x64', True) # required for ciMIST code to work properly
import cimist as cst
import matplotlib.pyplot as plt
import mdtraj as md
import pandas as pd
from itertools import islice

# ciMIST analysis guide
Run this notebook from the output directory. The model below loads the complete
saved candidate graph, including all empirical pair probabilities. Its memory
use grows with the number of analyzed residues and occupied state pairs. The
last example reads an individual saved pair without loading that graph.

For the theory, see [the ciMIST paper](https://www.biorxiv.org/content/10.1101/2025.05.28.656549v3).


In [ ]:
tree = cst.io.load_h5_tree("RESULTS_ciMIST.h5")
structure = md.load("pymol/structure.pdb")

# uncomment the following line to create a directory with a new pymol visualization
# tree.to_pymol(structure, "new_pymol")

In [ ]:
# estimated entropy, in nats
tree.entropy()

In [ ]:
# 95% credible interval for the entropy, in nats
tree.entropy_ci()

In [ ]:
# sum of residue marginal entropies, 
tree.sum_marginal_entropy()

In [ ]:
# sum of mutual informations
tree.sum_MIs()

In [ ]:
# marginal Shannon entropies of residues
tree.residue_entropies()

In [ ]:
# summed mutual informations of all neighbors for each residue
tree.residue_sum_MIs()

In [ ]:
# Eq. 3 allocations within the loaded model, in nats.
# A sum over selected residues is not their subsystem marginal entropy.
tree.residue_entropies() - 0.5 * tree.residue_sum_MIs()

In [ ]:
# mutual informations for all tree edges
tree.tree_MIs()

In [ ]:
# A networkx graph representing the tree
tree.T

In [ ]:
# The attribute I_pos_mean contains the mutual informations used in the entropy estimation
tree.T.edges(data="I_pos_mean")

In [ ]:
# The node attribute S_pos_mean contains the posterior mean marginal entropies.
tree.T.nodes(data="S_pos_mean")

In [ ]:
# Preview empirical mutual information for at most 20 candidate pairs.
# Avoid displaying every edge's probability array for a large system.
pd.DataFrame([(u, v, attrs["I"])
              for u, v, attrs in islice(tree.MI_graph.edges(data=True), 20)],
             columns=["residue_1", "residue_2", "empirical_MI_nats"])

## Select a marginal subsystem
Original topology indices distinguish repeated residue numbers across chains.
This table contains only the residue centers analyzed in this saved model.
Residues omitted during its original inference cannot be recovered by refitting.


In [ ]:
identity_fields = ("residue_index", "chain_index", "chain_id", "resSeq", "name")
identities = pd.DataFrame([{field: tree.nodes[node][field] for field in identity_fields}
                           for node in tree.resnames], index=tree.resnames)
identities.index.name = "label"
identities.head(20)

The example below keeps the first ten analyzed residues per chain, in original
order. Replace this technical example with a selection justified for your system.
For example, combine `identities["chain_index"].isin([0, 1])` with
`identities["resSeq"].between(10, 30)` for ranges in specific chains.

Refitting reuses the saved empirical pair probabilities, evaluates
all selected intrachain and interchain pairs, and rebuilds the tree and its
uncertainty with the same sample count, prior and entropy estimator. It leaves
`tree` unchanged. Excluded residues remain the physical environment sampled in
the simulation: their effects are averaged over, not conditioned away.


In [ ]:
selected = identities.index[identities.groupby("chain_index", sort=False).cumcount() < 10].tolist()
subset = tree.refit_to_subset(selected)
print(f"{len(selected)} residues; marginal tree entropy = "
      f"{float(subset.entropy()):.6f} +/- {float(subset.entropy_se):.6f} nats (standard error)")
# Uncomment to save the refitted model or visualize its new tree.
# cst.io.save_tree_h5(subset, "selected_tree.h5")
# subset.to_pymol(structure, "subset_view")

In [ ]:
# A bounded matrix view: empirical MI, with marginal entropy on the diagonal.
# Selecting matrix rows alone does not refit a subsystem tree.
matrix_names = subset.resnames[:20]
pd.DataFrame(subset.information_matrix(resis=matrix_names, diag=True),
             index=matrix_names, columns=matrix_names)

## Display selected residues within the existing tree
A focus view retains the selected nodes and excluded connector nodes on their
paths through the loaded tree. It preserves that model's statistics. Its
`focus.json` labels selected and connector identities and full-model Eq. 3
allocations. The sum below is the selected residues' allocated share of the
loaded model's entropy, distinct from the refitted marginal entropy above.


In [ ]:
# Uncomment to export this display-only view with its connector nodes.
# tree.to_pymol(structure, "focus_view", focus_nodes=selected)
full_model_allocations = tree.residue_entropies() - 0.5 * tree.residue_sum_MIs()
print(f"Selected residues' allocated share of the loaded model = "
      f"{float(full_model_allocations.loc[selected].sum()):.6f} nats")

## Read a saved pair with bounded memory
This example needs only the imports and the HDF5 file; it does not require
`load_h5_tree`. The returned record preserves the saved probability-axis order.
A singleton has no pair table. Use `saved.read_pairs([(label_a, label_b)])` to
request a particular pair by its exported labels.


In [ ]:
with cst.io.H5ProbabilityReader("RESULTS_ciMIST.h5") as saved:
    pair_preview = saved.read_rows([0]) if saved.n_edges else []
[(record.edge, record.axes, record.probability.dense_numpy()) for record in pair_preview]